In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B：同图中心化的可选CPU复核（无需GPU）

本地已运行且独立复核，用户无需重复运行；此入口仅方便重放。已填入fit-20260909-152025-689961的labels.jsonl和旧allocator.json。固定三特征、ridge=1，8折按图留一；每fold仅7图训练标准化/拟合，测试图仅自身特征中心化。4096次组内标签置换重跑同一拟合评价流程，不读validation24、不导出新allocator。

已观测：同图中心化平均排序rho由−.075升至.225，但最佳区域命中3/8降至2/8，选块中心化utility从+.015965降至+.006486。效果混合，不形成新GPU候选，不据此否定B方向；置换仅背景，不设置p值硬门槛。

只需要CPU和Drive挂载，不读取HF或水印密钥。输出诊断JSON/NPZ至Drive独立cpu-region-loo-时间目录，保留旧资产。无需修改参数。


In [ ]:
import json, os, pathlib, subprocess, sys, datetime
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='dev/survival-allocator-v1'
EXPECTED_EXACT='97eefe70e238dda9aff359348ba93db1b50585b6'
checkout=pathlib.Path('/content/survival-allocator-v1-github')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/survival-allocator-v1')
INPUT=drive_root/'fit-20260909-152025-689961'
output_root=drive_root/('cpu-region-loo-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
if not checkout.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
environment={}
for package in ('numpy',):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'branch':BRANCH,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
command=[sys.executable,'-m','experiments.run_survival_region_loo',
         '--input',str(INPUT),'--output',str(output_root)]
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
print('输出目录:', output_root)
if completed.returncode != 0:
    raise RuntimeError(f'诊断退出码 {completed.returncode}；已写入结果保留在 {output_root}')
report_path=output_root/'loo_region_centering_result.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print({'report':str(report_path),'summary':report.get('results',report.get('path_comparisons'))})
